# 한국어 대화 요약 - Qwen3-9B LoRA SFT (Unsloth)

## 개요
- **모델**: Qwen3-9B (14B 대비 GPU 효율 우수)
- **방법**: LoRA 파인튜닝 + Response-Only Loss
- **평가**: MeCab 형태소 기반 ROUGE (대회 공식 기준)
- **데이터 경로**: `/data/ephemeral/home/data/`


## 1. 환경 설정
필요한 패키지를 설치합니다.

In [1]:
# 처음 한 번만 실행
!pip install unsloth peft trl rouge datasets pandas wandb python-mecab-ko -q
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev -q


Reading package lists...
Building dependency tree...
Reading state information...
libmecab-dev is already the newest version (0.996-10build1).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-2.1).
mecab is already the newest version (0.996-10build1).
0 upgraded, 0 newly installed, 0 to remove and 46 not upgraded.


In [ ]:
import os
import re
import random, time
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import Dataset
from rouge import Rouge
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List

random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3090
VRAM: 25.4 GB


## 2. 하이퍼파라미터 설정

| 항목 | 14B | **9B (현재)** |
|------|-----|-------------|
| MODEL_NAME | unsloth/Qwen3-14B | **unsloth/Qwen3-9B** |
| PER_DEVICE_BATCH | 1 | **2** |
| GRAD_ACCUM | 32 | **16** |
| Effective Batch | 32 | **32 (동일)** |

⭐ `EXP_NAME`만 실험마다 변경하면 됩니다.


In [3]:
# ============================================================
# ⭐ 실험마다 변경할 것
# ============================================================
EXP_NAME = "qwen3_9b_lora_sft"   # 결과 저장 폴더명

# 모델 설정
MODEL_NAME     = "unsloth/Qwen3-8B"   # ✅ 14B → 9B 변경
MAX_SEQ_LENGTH = 1024    # ✅ 2048 → 1024 변경

# 데이터 경로 (서버 환경)
DATA_PATH   = "/data/ephemeral/home/data/"
RESULT_PATH = "/data/ephemeral/home/code/prediction/"

# LoRA 설정
LORA_R       = 16  # ✅ 32 → 16 변경
LORA_ALPHA   = 16  # ✅ 32 → 16 변경
LORA_DROPOUT = 0.0

# 학습 설정
LEARNING_RATE = 2e-4
EPOCHS        = 3
WARMUP_RATIO  = 0.05
WEIGHT_DECAY  = 0.01

# ✅ 9B는 메모리 여유 → batch 2로 증가 (effective batch 동일 32)
PER_DEVICE_BATCH = 1  # ✅ 2 → 1 변경
GRAD_ACCUM       = 8  # ✅ 16 → 8 변경

# 추론 설정
MAX_NEW_TOKENS = 128  # ✅ 192 → 128 변경

# wandb 설정 (원하면 활성화)
os.environ["WANDB_API_KEY"] = "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"
os.environ["WANDB_PROJECT"]  = "nlp-dialogue-summary"

print(f"Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Effective batch size: 8
GPU: NVIDIA GeForce RTX 3090


## 3. 데이터 로드

In [4]:
train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print(f"Train: {len(train_df):,}개")
print(f"Dev:   {len(dev_df):,}개")
print(f"Test:  {len(test_df):,}개")
print(f"컬럼: {train_df.columns.tolist()}")

# 기본 통계
print(f"\n[대화 길이] 평균: {train_df['dialogue'].str.len().mean():.0f}자  최대: {train_df['dialogue'].str.len().max()}자")
print(f"[요약 길이] 평균: {train_df['summary'].str.len().mean():.0f}자  최대: {train_df['summary'].str.len().max()}자")

# 태그 사용 비율 (EDA)
has_tag  = train_df['summary'].str.contains(r'#Person\d+#', regex=True).mean()
has_name = train_df['summary'].str.contains(r'\b[A-Z][a-z]+\b', regex=True).mean()
print(f"\n[요약 EDA]")
print(f"  #PersonN# 태그 포함 비율: {has_tag:.1%}")
print(f"  영문 이름 포함 비율     : {has_name:.1%}")


Train: 12,457개
Dev:   499개
Test:  499개
컬럼: ['fname', 'dialogue', 'summary', 'topic']

[대화 길이] 평균: 406자  최대: 2165자
[요약 길이] 평균: 86자  최대: 376자

[요약 EDA]
  #PersonN# 태그 포함 비율: 85.9%
  영문 이름 포함 비율     : 10.7%


## 4. 모델 로드 (4-bit 양자화)

- **원본 크기**: Qwen3-9B ≈ 18GB (FP16)
- **4-bit 양자화 후**: ≈ 5GB → RTX 3090에서도 충분히 실행 가능


In [5]:
# 1단계: torchvision 업그레이드
!pip install --upgrade "torchvision>=0.25.0" -q

# 2단계: 그래도 안되면 환경변수로 체크 스킵
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
print(f"모델 로드 완료: {MODEL_NAME}")


==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

unsloth/qwen3-8b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
모델 로드 완료: unsloth/Qwen3-8B


In [ ]:
import torch
torch.cuda.empty_cache()
torch.backends.cuda.matmul.allow_tf32 = True
print(f"남은 VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9 - torch.cuda.memory_allocated()/1e9:..1f}GB")

## 5. LoRA 어댑터 설정

전체 파라미터의 약 **1% 미만**만 학습 → 빠르고 메모리 효율적


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 가능 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


학습 가능 파라미터: 43,646,976 / 5,235,454,976 (0.83%)


## 6. 프롬프트 설계

- system: 태그 유지 + 간결 요약 지시
- user: 대화 원문
- assistant: 정답 요약 (학습 시에만)


In [ ]:
SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다. "
    "#Person1#, #Person2# 등 화자 태그를 절대 바꾸지 말고 그대로 유지하세요. "
    "이름(Jimmy, Karen 등)이나 역할(의사, 환자 등)로 대체하지 마세요. "
    "대화의 핵심 내용을 1개만 포함시켜, 한국어 문어체로 반드시 20단어 이하의 1문장으로 요약하세요."
    "불필요한 세부 설명은 포함하지 않습니다."
)

USER_TEMPLATE = (
    "아래 대화를 읽고 한국어 문어체로 #Person 태그를 유지하며 핵심 내용을 1개만 포함시켜 20단어 이하의 1문장으로 요약하세요.\n\n{dialogue}"
)


def create_messages(dialogue, summary=None, is_train=True):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_TEMPLATE.format(dialogue=dialogue)},
    ]
    if is_train and summary is not None:
        messages.append({"role": "assistant", "content": str(summary)})
    return messages


# 샘플 확인
sample_text = tokenizer.apply_chat_template(
    create_messages(train_df.iloc[0]["dialogue"], train_df.iloc[0]["summary"]),
    tokenize=False, add_generation_prompt=False, enable_thinking=False
)
print("[채팅 템플릿 예시 (앞 400자)]")
print(sample_text[:400])
print("...")
print(sample_text[-150:])


[채팅 템플릿 예시 (앞 400자)]
<|im_start|>system
당신은 한국어 대화 요약 전문가입니다. #Person1#, #Person2# 등 화자 태그를 절대 바꾸지 말고 그대로 유지하세요. 이름(Jimmy, Karen 등)이나 역할(의사, 환자 등)로 대체하지 마세요. 핵심 내용만 1~2문장으로 간결하게 요약하세요.<|im_end|>
<|im_start|>user
아래 대화를 읽고 #Person 태그를 유지하며 핵심 내용을 요약하세요.

#Person1#: 안녕하세요, Mr. Smith. 저는 Dr. Hawkins입니다. 오늘 무슨 일로 오셨어요? 
#Person2#: 건강검진을 받으려고 왔어요. 
#Person1#: 네, 5년 동안 검진을 안 받으셨네요. 매년 한 번씩 받으셔야 해요. 
#Person2#: 알죠. 특별히 아픈 데가
...
의사 선생님.<|im_end|>
<|im_start|>assistant
<think>

</think>

Mr. Smith는 Dr. Hawkins에게 건강검진을 받으러 와서, 매년 검진 필요성을 안내받고 흡연 습관 개선을 위한 도움을 제안받았습니다.<|im_end|>



## 7. Response-Only Loss

프롬프트 부분은 loss 계산 제외 → **요약 생성에만 집중**

```
[시스템 프롬프트] → loss X (마스킹)
[사용자 입력]     → loss X (마스킹)
[모델 응답 요약]  → loss O ← 이것만 학습!
```


In [10]:
@dataclass
class ResponseOnlyDataCollator:
    tokenizer: Any
    response_template_ids: List[int] = None
    max_length: int = 2048

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(
                features, padding=True,
                max_length=self.max_length, return_tensors="pt"
            )
        else:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts, return_tensors="pt", padding=True,
                truncation=True, max_length=self.max_length
            )

        labels = batch["input_ids"].clone()
        for i in range(len(labels)):
            ids = batch["input_ids"][i].tolist()
            response_start = self._find_response_start(ids)
            if response_start >= 0:
                labels[i, :response_start] = -100
            labels[i, batch["attention_mask"][i] == 0] = -100
        batch["labels"] = labels
        return batch

    def _find_response_start(self, ids: List[int]) -> int:
        template = self.response_template_ids
        if template is None:
            return 0
        last_pos = -1
        for i in range(len(ids) - len(template) + 1):
            if ids[i:i + len(template)] == template:
                last_pos = i + len(template)
        return last_pos


# Response template 설정
response_template_str = "<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(response_template_str, add_special_tokens=False)

collator = ResponseOnlyDataCollator(
    tokenizer=tokenizer,
    response_template_ids=response_template_ids,
    max_length=MAX_SEQ_LENGTH,
)

# 검증
sample_ids = tokenizer.encode(sample_text, add_special_tokens=False)
pos = collator._find_response_start(sample_ids)
print(f"Response template IDs: {response_template_ids}")
print(f"검증: {pos}/{len(sample_ids)} 토큰 마스킹 ({pos/len(sample_ids)*100:.1f}%)")
print("→ 프롬프트 부분은 학습에서 제외됩니다!")


Response template IDs: [151644, 77091, 198]
검증: 461/517 토큰 마스킹 (89.2%)
→ 프롬프트 부분은 학습에서 제외됩니다!


## 8. 학습 데이터 준비

In [11]:
def formatting_prompts_func(examples):
    texts = []
    for dialogue, summary in zip(examples["dialogue"], examples["summary"]):
        messages = create_messages(dialogue, summary, is_train=True)
        text = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=False, enable_thinking=False,
        )
        texts.append(text)
    return {"text": texts}


train_dataset = Dataset.from_pandas(train_df[["dialogue", "summary"]]).map(
    formatting_prompts_func, batched=True
)
dev_dataset = Dataset.from_pandas(dev_df[["dialogue", "summary"]]).map(
    formatting_prompts_func, batched=True
)

print(f"학습 데이터: {len(train_dataset):,}개")
print(f"검증 데이터: {len(dev_dataset):,}개")


Map:   0%|          | 0/12457 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

학습 데이터: 12,457개
검증 데이터: 499개


## 9. 학습 (SFT)

예상 학습 시간:
- RTX 3090 (24GB): 약 2~3시간
- A100 (80GB): 약 1시간


In [12]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

OUTPUT_DIR = f"/data/ephemeral/home/code/outputs/{EXP_NAME}"

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,

    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    seed=3407,
    output_dir=OUTPUT_DIR,
    report_to="wandb",   # wandb 끄려면 "none"으로 변경
    optim="adamw_8bit",
    max_grad_norm=1.0,
    run_name=EXP_NAME,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    args=sft_config,
    data_collator=collator,
)

print(f"학습 준비 완료!")
print(f"  Effective batch size : {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"  총 스텝 수           : {len(train_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM) * EPOCHS}")
print(f"  BF16                 : {is_bfloat16_supported()}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/12457 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/499 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


학습 준비 완료!
  Effective batch size : 8
  총 스텝 수           : 4671
  BF16                 : True


In [13]:
# ★ 학습 시작!
print(f"[{EXP_NAME}] 학습 시작...")
trainer_stats = trainer.train()

print(f"\n✅ 학습 완료!")
print(f"  총 학습 시간  : {trainer_stats.metrics['train_runtime']:.0f}초")
print(f"  최종 train loss: {trainer_stats.metrics['train_loss']:.4f}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


[qwen3_9b_lora_sft] 학습 시작...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,457 | Num Epochs = 3 | Total steps = 4,674
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: s202010741 (s202010741-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Epoch,Training Loss,Validation Loss
1,0.665165,0.663431
2,0.587254,0.657434
3,0.386108,0.726180


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 1100, in emit
    msg = self.format(record)
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 943, in format
    return fmt.format(record)
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 678, in format
    record.message = record.getMessage()
  File "/opt/conda/lib/python3.10/logging/__init__.py", line 368, in getMessage
    msg = msg % self.args
TypeError: not all arguments converted during string formatting
Call stack:
  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code

eval/loss,▂▁█
eval/runtime,█▃▁
eval/samples_per_second,▁▆█
eval/steps_per_second,▁▆█
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇█████
train/grad_norm,▇▂▃▁▂▃▂▂▂▂▂▂▂▂▁▂▃▃▃▂▃▂▃▂▃▂▄▄▅▄▇▆▄▅▅█▄▅▇▅
train/learning_rate,▂▅████████▇▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▂▂▂▂▁▁▁▁▁
train/loss,▇█▆▆▆▆▆▅▆▇▄▄▄▅▄▄▄▄▅▃▄▄▄▄▄▁▁▁▁▁▂▂▁▁▁▂▁▂▂▂
eval/loss,0.72618
eval/runtime,82.9723



✅ 학습 완료!
  총 학습 시간  : 19844초
  최종 train loss: 0.5778


In [14]:
# LoRA 어댑터 저장
LORA_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"✅ LoRA 어댑터 저장 완료: {LORA_PATH}")


✅ LoRA 어댑터 저장 완료: /data/ephemeral/home/code/outputs/qwen3_9b_lora_sft/lora_adapter


## 10. 추론 (Inference)

In [15]:
# 추론 모드 전환
FastLanguageModel.for_inference(model)


def postprocess(text):
    """생성된 요약 후처리"""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    return text if text else "빈 요약"


def generate_summary(dialogue):
    """대화 → 요약 생성"""
    messages = create_messages(dialogue, is_train=False)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return postprocess(summary)


# 추론 예시 (dev 3개)
print("[추론 예시 확인]")
for i in range(3):
    pred = generate_summary(dev_df.iloc[i]["dialogue"])
    gold = dev_df.iloc[i]["summary"]
    print(f"\n--- 예시 {i+1} ---")
    print(f"  정답: {gold}")
    print(f"  예측: {pred}")


[추론 예시 확인]

--- 예시 1 ---
  정답: #Person2#는 숨쉬기 어려워합니다. 의사는 #Person2#에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
  예측: #Person2#는 숨쉬기가 힘들다고 하며, #Person1#은 천식 검사를 위해 폐 전문의를 추천합니다.

--- 예시 2 ---
  정답: #Person1#는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
  예측: #Person1#과 #Person2#는 운동 계획을 조정하며, #Person1#은 팔과 복근 운동을 원하고 #Person2#는 다리 운동을 원합니다.

--- 예시 3 ---
  정답: #Person1#은 건강에 안 좋은 음식을 그만 먹기로 결심하고, #Person2#는 자신의 건강한 식단을 #Person1#에게 공유합니다.
  예측: #Person1#과 #Person2#는 건강한 식습관을 유지하기 위해 과일, 채소, 닭고기를 섭취합니다.


## 11. Dev Set ROUGE 평가

In [16]:
# Dev set 전체 추론
print(f"Dev set 추론 시작 ({len(dev_df)}개)...")
dev_preds = []
for i in tqdm(range(len(dev_df))):
    pred = generate_summary(dev_df.iloc[i]["dialogue"])
    dev_preds.append(pred)
print(f"추론 완료: {len(dev_preds)}개")


Dev set 추론 시작 (499개)...


  0%|          | 0/499 [00:00<?, ?it/s]

추론 완료: 499개


In [17]:
# ROUGE 평가 (공백 기반 + MeCab 기반)
rouge = Rouge()
golds = [str(s).strip() if str(s).strip() else "빈 요약" for s in dev_df["summary"]]
preds = [p if p.strip() else "빈 요약" for p in dev_preds]

# 공백 기반
scores = rouge.get_scores(preds, golds, avg=True)
print("=" * 50)
print(f"[{EXP_NAME}] Dev ROUGE (공백 기반):")
print(f"  ROUGE-1: {scores['rouge-1']['f']:.4f}")
print(f"  ROUGE-2: {scores['rouge-2']['f']:.4f}")
print(f"  ROUGE-L: {scores['rouge-l']['f']:.4f}")
avg_space = (scores['rouge-1']['f'] + scores['rouge-2']['f'] + scores['rouge-l']['f']) / 3
print(f"  AVG    : {avg_space:.4f}")
print("=" * 50)

# MeCab 기반 (대회 공식)
try:
    from mecab import MeCab
    m = MeCab()
    golds_tok = [" ".join([t for t, _ in m.pos(g) if t.strip()]) for g in golds]
    preds_tok = [" ".join([t for t, _ in m.pos(p) if t.strip()]) for p in preds]

    mecab_scores = rouge.get_scores(preds_tok, golds_tok, avg=True)
    print(f"[{EXP_NAME}] Dev ROUGE (MeCab 형태소 - 대회 공식):")
    print(f"  ROUGE-1: {mecab_scores['rouge-1']['f']:.4f}")
    print(f"  ROUGE-2: {mecab_scores['rouge-2']['f']:.4f}")
    print(f"  ROUGE-L: {mecab_scores['rouge-l']['f']:.4f}")
    avg_mecab = (mecab_scores['rouge-1']['f'] + mecab_scores['rouge-2']['f'] + mecab_scores['rouge-l']['f']) / 3
    print(f"  AVG    : {avg_mecab:.4f}  ← 리더보드 점수와 유사")
    print("=" * 50)
except Exception as e:
    print(f"MeCab 오류: {e}")


[qwen3_9b_lora_sft] Dev ROUGE (공백 기반):
  ROUGE-1: 0.3143
  ROUGE-2: 0.1270
  ROUGE-L: 0.2989
  AVG    : 0.2468
[qwen3_9b_lora_sft] Dev ROUGE (MeCab 형태소 - 대회 공식):
  ROUGE-1: 0.5566
  ROUGE-2: 0.3656
  ROUGE-L: 0.4930
  AVG    : 0.4717  ← 리더보드 점수와 유사


## 12. Test Set 추론 & 제출 파일 생성

In [18]:
# Test set 추론
print(f"Test set 추론 시작 ({len(test_df)}개)...")
test_preds = []
for i in tqdm(range(len(test_df))):
    pred = generate_summary(test_df.iloc[i]["dialogue"])
    test_preds.append(pred)
print(f"추론 완료: {len(test_preds)}개")


Test set 추론 시작 (499개)...


  0%|          | 0/499 [00:00<?, ?it/s]

추론 완료: 499개


In [19]:
# 제출 파일 생성
os.makedirs(RESULT_PATH, exist_ok=True)

submission = pd.DataFrame({
    "fname"  : test_df["fname"],
    "summary": test_preds,
})

sub_path = os.path.join(RESULT_PATH, f"submission_{EXP_NAME}.csv")
submission.to_csv(sub_path, index=False)

print(f"✅ 제출 파일 저장: {sub_path}")
print(f"총 {len(submission)}개 행")

# 통계
print(f"\n[생성 요약 통계]")
print(f"  평균 길이    : {submission['summary'].str.len().mean():.0f}자")
print(f"  #Person 포함 : {submission['summary'].str.contains('#Person', regex=False).mean():.1%}")
print(f"\n[미리보기]")
print(submission.head(5).to_string())


✅ 제출 파일 저장: /data/ephemeral/home/code/prediction/submission_qwen3_9b_lora_sft.csv
총 499개 행

[생성 요약 통계]
  평균 길이    : 88자
  #Person 포함 : 86.4%

[미리보기]
    fname                                                                                                                                      summary
0  test_0                               #Person1#은 Ms. Dawson에게 모든 직원에게 사내 메모를 보내달라고 요청합니다. #Person1#은 즉시 메시지 프로그램 사용을 금지하고, 두 번 위반 시 해고될 수 있다고 설명합니다.
1  test_1  #Person2#는 교통체증으로 인해 지연되었고, #Person1#은 대중교통을 이용하는 것을 제안합니다. #Person2#는 대중교통이 좋다고 생각하지만, 차가 줄 수 있는 자유를 그리워합니다. #Person1#은 자전거로 출근할 것을 제안합니다.
2  test_2                                                         #Person1#은 Kate에게 Masha와 Hero가 이혼 신청을 했다고 말합니다. Kate는 놀라며 그들이 완벽한 커플이었기 때문이라고 생각합니다.
3  test_3                                                                             #Person1#은 Brian의 생일 파티에서 그를 축하하며 춤을 청하고, Brian은 그녀의 드레스를 칭찬합니다.
4  test_4                                                                                    #Pe

## 13. 저장된 LoRA 어댑터 재로드 (새 세션에서 사용)

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,   # 저장된 경로
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```
